# Stage 14: cross-fitted learned emission residual

CPU-only validation notebook. It trains fold-local HGB residual models over target-invariant emission diagnostics, then nested-selects conservative correction weights and caps. It creates no Kaggle submission.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess

REPOSITORY_URL = 'https://github.com/Okada-N13/rogii.git'
repo_dir = Path('/content/ROGII')
drive_root = Path('/content/drive/MyDrive/kaggle/rogii')
artifact_dir = drive_root / 'artifacts'
if not (repo_dir / '.git').is_dir():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(repo_dir)], check=True)
else:
    subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only', 'origin', 'main'], check=True)
if shutil.which('uv') is None:
    subprocess.run(['bash', '-lc', 'curl -LsSf https://astral.sh/uv/install.sh | sh'], check=True)
os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']
subprocess.run(['uv', 'sync', '--frozen'], cwd=repo_dir, check=True)
artifact_dir.mkdir(parents=True, exist_ok=True)
subprocess.run(['git', '-C', str(repo_dir), 'rev-parse', '--short', 'HEAD'], check=True)


## Verify prerequisites

Stage 14 reuses the expensive Stage 12B/12C predictions and the Stage 13 decision. Run notebooks 280–300 first if these Drive artifacts are missing.


In [ ]:
STAGE12B_RUN_ID = 'stage12b_learned_emission_tcn_full_v001'
STAGE12C_RUN_ID = 'stage12c_spatial_kbest_lattice_full_v001'
STAGE13_RUN_ID = 'stage13_emission_uncertainty_gate_full_v001'
stage12b_run = artifact_dir / STAGE12B_RUN_ID
stage12c_run = artifact_dir / STAGE12C_RUN_ID
stage13_run = artifact_dir / STAGE13_RUN_ID
assert (stage12b_run / 'oof_emission.parquet').is_file(), 'Run notebook 280 first'
assert (stage12c_run / 'fold_path_oof.parquet').is_file(), 'Run notebook 290 first'
assert (stage13_run / 'gate_summary.json').is_file(), 'Run notebook 300 first'
assert json.loads((stage12b_run / 'gate_summary.json').read_text())['promoted_to_spatial_emission_audit'] is True
assert json.loads((stage13_run / 'gate_summary.json').read_text())['promoted_to_all_train_uncertainty_model'] is False
print('Prerequisites verified')


## Train cross-fitted residual models

Keep `LIMIT_WELLS = None`. Twenty-one small HGB models are trained on CPU. Completed fold checkpoints are reused after a disconnect.


In [ ]:
RUN_ID = 'stage14_crossfit_emission_residual_full_v001'
LIMIT_WELLS = None
run_dir = artifact_dir / RUN_ID
if not (run_dir / 'gate_summary.json').is_file():
    command = [
        'uv', 'run', 'rogii-emission-residual',
        '--config', 'configs/experiment/stage14_crossfit_emission_residual.yaml',
        '--stage12b-run', str(stage12b_run), '--stage12c-run', str(stage12c_run), '--stage13-run', str(stage13_run),
        '--artifact-dir', str(artifact_dir), '--run-id', RUN_ID, '--resume',
    ]
    if LIMIT_WELLS is not None:
        command += ['--limit-wells', str(LIMIT_WELLS)]
    log_path = artifact_dir / f'{RUN_ID}_driver.log'
    with log_path.open('w', encoding='utf-8') as log_handle:
        process = subprocess.Popen(command, cwd=repo_dir, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end=''); log_handle.write(line); log_handle.flush()
        return_code = process.wait()
    if return_code != 0:
        print('\n'.join(log_path.read_text(encoding='utf-8', errors='replace').splitlines()[-160:]))
        raise RuntimeError(f'Stage 14 failed with exit code {return_code}. Full log: {log_path}')
else:
    print('Reusing completed run:', run_dir)


## Decision

Generic models use the same feature schema in all three holdouts. The standard-only stacked model additionally uses branch disagreement and entropy.


In [ ]:
summary = json.loads((run_dir / 'gate_summary.json').read_text(encoding='utf-8'))
families = summary['family_reports']
decision = {
    'promoted_to_full_residual_training': summary['promoted_to_full_residual_training'],
    'standard_base_rmse': families['fold']['base_metrics']['pooled_rmse'],
    'standard_nested_rmse': families['fold']['nested_metrics']['pooled_rmse'],
    'standard_delta': families['fold']['nested_rmse_delta'],
    'spatial_base_rmse': families['spatial_fold']['base_metrics']['pooled_rmse'],
    'spatial_nested_rmse': families['spatial_fold']['nested_metrics']['pooled_rmse'],
    'spatial_delta': families['spatial_fold']['nested_rmse_delta'],
    'typewell_base_rmse': families['typewell_fold']['base_metrics']['pooled_rmse'],
    'typewell_nested_rmse': families['typewell_fold']['nested_metrics']['pooled_rmse'],
    'typewell_delta': families['typewell_fold']['nested_rmse_delta'],
    'bootstrap_95pct': {name: [report['bootstrap']['ci_2_5'], report['bootstrap']['ci_97_5']] for name, report in families.items()},
    'robust_generic_spec': summary['robust_generic_spec'],
    'gates': summary['gates'], 'next_step': summary['next_step'],
}
decision


In [ ]:
import pandas as pd
profile_rows = []
for family, report in families.items():
    for name, values in report['profile_reports'].items():
        profile_rows.append({'family': family, 'profile': name, 'rmse_delta': values['pooled_rmse_delta'], 'worst_fold_delta': max(values['fold_deltas'].values()), 'p90_delta': values['candidate_metrics']['well_rmse_p90'] - values['base_metrics']['well_rmse_p90'], 'worst10_delta': values['candidate_metrics']['worst_10pct_sse_share'] - values['base_metrics']['worst_10pct_sse_share']})
pd.DataFrame(profile_rows).sort_values(['family', 'rmse_delta']).reset_index(drop=True)


## Send back

Send the decision dictionary and profile table. No submission is produced.
